# Column with node-to-surface constraints — v2

Changes from v1:

- **Dead code removed** (unused `pg_points` and test `add_curve`).
- **Material fixed** — `E = 2**9 = 512 MPa` was a typo; now `E = 200e3 MPa` (steel, N/mm²).
- **DOF mismatch fixed** — OpenSees model is now `ndf=6` globally (so reference + phantom nodes default to 6 DOF and accept the 6-component `(Fx,Fy,Fz,Mx,My,Mz)` load record); the solid column nodes are overridden to `ndf=3`.
- **Self-contained OpenSees cell** — phantom node data is re-queried inside the build cell instead of depending on an earlier notebook cell.
- **Single constraint-iteration path** — uses the flat `rigid_link_groups()` / `equal_dofs()` API only.
- **Verification** — sums base reactions across the full constraint chain and compares the cantilever tip displacement against the Euler-Bernoulli and Timoshenko closed-form solutions.

In [1]:
from apeGmsh import apeGmsh, Part
from apeGmsh import FEMData
from apeGmsh.sections import W_solid
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
m1 = apeGmsh(model_name='beam_column', verbose=False)
m1.begin()

column = m1.sections.W_solid(bf=150, tf=20, h=300, tw=10, length=2000, label='column')

m1.model.selection.select_volumes().to_physical(name='pg_column')

base = m1.model.geometry.add_point(x=0, y=0, z=0,    lc=100, label='base')
top  = m1.model.geometry.add_point(x=0, y=0, z=2000, lc=100, label='top')

# Constraints: couple reference points to the column end faces
m1.constraints.node_to_surface('base', column.labels.start_face)
m1.constraints.node_to_surface('top',  column.labels.end_face)

# Lateral point load at top (Fy = 1000 N)
m1.loads.point(
    target='top',
    force_xyz=[0, 1000, 0],
    moment_xyz=[0, 0, 0],
)

m1.mesh.sizing.set_global_size(100)
m1.mesh.generation.generate(dim=3)

fem = m1.mesh.queries.get_fem_data(remove_orphans=True)

m1.model.viewer()
m1.mesh.viewer()

m1.end()

## Inspect the FEM broker output

In [3]:
# Solid column nodes + the two reference points
node = fem.nodes.get(target=['pg_column', 'base', 'top'])
nodes_df = pd.DataFrame(node.coords, columns=['x', 'y', 'z'])
nodes_df.insert(0, 'id', node.ids)
nodes_df

,id,x,y,z
0,1,-7.500000e+01,-150.000000,0.000000
1,2,-7.500000e+01,-150.000000,2000.000000
2,3,-7.500000e+01,-170.000000,0.000000
3,4,-7.500000e+01,-170.000000,2000.000000
4,5,-5.000000e+00,-150.000000,0.000000
...,...,...,...,...
641,644,2.273737e-17,-163.333333,833.333333
642,645,9.094947e-17,-163.333333,1433.333333
643,646,0.000000e+00,-160.000000,1350.000000
644,33,0.000000e+00,0.000000,0.000000


In [ ]:
# Phantom nodes generated by the node-to-surface constraints
phantom = fem.nodes.constraints.phantom_nodes()

phantom_df = pd.DataFrame(phantom.coords, columns=['x', 'y', 'z'])
phantom_df.insert(0, 'id', phantom.ids)
phantom_df

In [5]:
# tet4 connectivity for the column volume
element_ids, element_connectivity = fem.elements.get(target='pg_column').resolve()

elements_df = pd.DataFrame(element_connectivity)
elements_df.insert(0, 'id', element_ids)
elements_df

,id,0,1,2,3
0,47,352,58,57,368
1,48,43,42,356,374
2,49,96,97,384,362
3,50,372,40,60,354
4,51,351,38,37,386
...,...,...,...,...,...
1785,1832,187,188,90,261
1786,1833,189,88,89,260
1787,1834,89,88,189,94
1788,1835,89,188,189,260


In [6]:
# Flat constraint iteration — what the OpenSees build cell will consume
print('Rigid links:')
for master, slaves in fem.nodes.constraints.rigid_link_groups():
    print(f'  master {master} -> {len(slaves)} phantom nodes')

print('\nEqual DOFs:')
eq_pairs = list(fem.nodes.constraints.equal_dofs())
for pair in eq_pairs[:4]:
    print(f'  {pair.master_node} -> {pair.slave_node}  dofs={pair.dofs}')
print(f'  ... ({len(eq_pairs)} total)')

Rigid links:
  master 33 -> 22 phantom nodes
  master 34 -> 22 phantom nodes

Equal DOFs:
  647 -> 1  dofs=[1, 2, 3]
  648 -> 3  dofs=[1, 2, 3]
  649 -> 5  dofs=[1, 2, 3]
  650 -> 7  dofs=[1, 2, 3]
  ... (44 total)


## Build and run the OpenSees model

DOF layout:

- Global `ndf = 6` so reference points (`base`, `top`) and phantom nodes inherit 6 DOFs by default — loads from `NodalLoadRecord.forces` are 6-tuples `(Fx,Fy,Fz,Mx,My,Mz)` and must match.
- Solid column nodes are overridden to `ndf = 3` (translations only) to match `FourNodeTetrahedron`.
- `equalDOF(phantom_6dof, solid_3dof, 1, 2, 3)` only constrains translations, which is the shared DOF set.

Constraint handler: `Penalty`. `Transformation` cannot be used here because the constraint chain is daisy-chained — each phantom node is both a `rigidLink` slave (to a reference point) *and* an `equalDOF` master (to a solid face node), and Transformation requires each constrained DOF to appear in at most one constraint. `Lagrange` also works but introduces extra DOFs; `Penalty` is sufficient.

In [ ]:
import openseespy.opensees as ops

ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)

linear_time_series_tag = 1
ops.timeSeries('Linear', linear_time_series_tag)

# --- Nodes -----------------------------------------------------------------
# Solid column nodes -> 3 DOF
col = fem.nodes.get(target='pg_column')
for nid, xyz in col:
    ops.node(nid, *xyz, '-ndf', 3)

# Reference points (base, top) -> default 6 DOF
ref = fem.nodes.get(target=['base', 'top'])
for nid, xyz in ref:
    ops.node(nid, *xyz)

# Phantom nodes (from node-to-surface constraints) -> default 6 DOF
phantoms = fem.nodes.constraints.phantom_nodes()
for nid, xyz in phantoms:
    ops.node(nid, *xyz)

# --- Boundary conditions ---------------------------------------------------
for base_id in fem.nodes.get_ids(target='base'):
    ops.fix(int(base_id), 1, 1, 1, 1, 1, 1)

# --- Material + elements ---------------------------------------------------
E  = 200e3   # steel, N/mm^2
nu = 0.3
material_tag = 1
ops.nDMaterial('ElasticIsotropic', material_tag, E, nu)

for group in fem.elements.get(element_type='tet4'):
    for eid, conn in group:
        ops.element('FourNodeTetrahedron', int(eid), *[int(n) for n in conn], material_tag)

# --- Constraints -----------------------------------------------------------
for master, slaves in fem.nodes.constraints.rigid_link_groups():
    for slave in slaves:
        ops.rigidLink('beam', int(master), int(slave))

for pair in fem.nodes.constraints.equal_dofs():
    ops.equalDOF(int(pair.master_node), int(pair.slave_node), *pair.dofs)

# --- Loads -----------------------------------------------------------------
load_pattern_tag = 1
ops.pattern('Plain', load_pattern_tag, linear_time_series_tag)
for load in fem.nodes.loads.by_kind('nodal'):
    fx, fy, fz = load.force_xyz  or (0.0, 0.0, 0.0)
    mx, my, mz = load.moment_xyz or (0.0, 0.0, 0.0)
    ops.load(int(load.node_id), fx, fy, fz, mx, my, mz)

# --- Analysis --------------------------------------------------------------
ops.constraints('Penalty', 1e15, 1e15)
ops.numberer('RCM')
ops.system('UmfPack')
ops.test('NormDispIncr', 1.0e-6, 10)
ops.algorithm('Linear')
ops.integrator('LoadControl', 0.1)
ops.analysis('Static')

for i in range(10):
    ok = ops.analyze(1)
    print(f'Step {i+1}: {"ok" if ok == 0 else f"failed ({ok})"}')

## Verify base reaction

Applied load: `F = (0, 1000, 0) N` at the top reference point (`z = 2000 mm`). Taking moments about the origin, the equilibrium reaction at `z = 0` is:

- `ΣFy = -1000 N`
- `ΣMx = -(r × F)_x = -(2000 · 1000) + ... = +2·10⁶ N·mm`  (the applied force produces `Mx_applied = -2e6`, so the base must resist with `+2e6`)

`ops.nodeReaction(33)` on the base reference point returns **zeros**, because node 33 sits at the top of a `rigidLink` → `equalDOF` chain and OpenSees's per-node reaction output does not aggregate MP-constraint flow-through. To check equilibrium we sum nodal reactions across **every node at z = 0** — the ref point, the base phantoms, and the solid base-face nodes.

In [ ]:
ops.reactions()

all_ids   = list(col.ids) + list(ref.ids) + list(phantoms.ids)
all_xyz   = list(col.coords) + list(ref.coords) + list(phantoms.coords)

total_F = np.zeros(3)
total_M = np.zeros(3)  # moment about origin
for nid, xyz in zip(all_ids, all_xyz):
    if abs(xyz[2]) > 1e-6:
        continue
    r = ops.nodeReaction(int(nid))
    if not r:
        continue
    f = np.array([r[0], r[1], r[2]])
    total_F += f
    total_M += np.cross(np.array(xyz), f)
    if len(r) == 6:
        total_M += np.array([r[3], r[4], r[5]])

expected_Fy = -1000.0
expected_Mx = +2.0e6

print('Summed base reaction (all nodes at z = 0):')
print(f'  Fx = {total_F[0]:+.4e}   Fy = {total_F[1]:+.4e}   Fz = {total_F[2]:+.4e}')
print(f'  Mx = {total_M[0]:+.4e}   My = {total_M[1]:+.4e}   Mz = {total_M[2]:+.4e}')
print(f'\nExpected:  Fy = {expected_Fy:+.4e}   Mx = {expected_Mx:+.4e}')
print(f'Error:     Fy = {total_F[1] - expected_Fy:+.3e}   Mx = {total_M[0] - expected_Mx:+.3e}')

## Verify tip displacement against cantilever closed-form

The W-section is extruded along `z` with its web lying in the `y`-direction, so bending due to `Fy` is about the **strong (x) axis**. For the W-section `bf=150, tf=20, h=300, tw=10` (with `h` defined as the clear web height):

`Ix = bf·H³/12 − (bf − tw)·h³/12`  where `H = h + 2·tf = 340 mm`

- **Euler-Bernoulli:**  `δ_EB = P·L³ / (3·E·Ix)`
- **Timoshenko** (adds shear compliance, `As ≈ tw·h` for a W-section strong-axis web):  `δ_T = δ_EB + P·L / (G·As)`,  with `G = E / (2(1+ν))`

A tet4 solid mesh captures both bending and shear, so the FEM result should fall between the two closed-form bounds.

In [9]:
top_id = int(next(iter(fem.nodes.get_ids(target='top'))))
disp = ops.nodeDisp(top_id)

# Closed-form cantilever (strong-axis bending about x)
bf, tf, h_web, tw, L = 150.0, 20.0, 300.0, 10.0, 2000.0
H  = 2 * tf + h_web                                  # total depth = 340 mm
Ix = (bf * H**3) / 12 - ((bf - tw) * h_web**3) / 12
P  = 1000.0

delta_EB    = P * L**3 / (3 * E * Ix)
G           = E / (2 * (1 + nu))
As          = tw * h_web                              # web shear area (strong axis)
delta_shear = P * L / (G * As)
delta_T     = delta_EB + delta_shear

print(f'Top node {top_id} displacement:')
print(f'  ux = {disp[0]:+.4e}   uy = {disp[1]:+.4e}   uz = {disp[2]:+.4e}')
print(f'\nSection properties:')
print(f'  Ix = {Ix:.4e} mm^4   As = {As:.1f} mm^2')
print(f'\nCantilever closed-form tip deflection:')
print(f'  delta_EB         = {delta_EB:.4e} mm   (bending only)')
print(f'  delta_shear      = {delta_shear:.4e} mm')
print(f'  delta_Timoshenko = {delta_T:.4e} mm')
print(f'  FEM uy           = {disp[1]:+.4e} mm')
print(f'\nRatios:')
print(f'  FEM / Euler-Bernoulli = {disp[1] / delta_EB:.4f}')
print(f'  FEM / Timoshenko      = {disp[1] / delta_T:.4f}')

Top node 34 displacement:
  ux = -1.6524e-04   uy = +7.8671e-02   uz = +2.4289e-06

Section properties:
  Ix = 1.7630e+08 mm^4   As = 3000.0 mm^2

Cantilever closed-form tip deflection:
  delta_EB         = 7.5629e-02 mm   (bending only)
  delta_shear      = 8.6667e-03 mm
  delta_Timoshenko = 8.4295e-02 mm
  FEM uy           = +7.8671e-02 mm

Ratios:
  FEM / Euler-Bernoulli = 1.0402
  FEM / Timoshenko      = 0.9333
